# Daten laden: King County House Sales

Dieses Notebook lädt die Rohdaten aus der PostgreSQL-Datenbank und speichert sie als `data/eda.csv`. Die Analyse folgt in [02_eda.ipynb](02_eda.ipynb).

Die Daten liegen im Schema `eda` in zwei Tabellen:
- `king_county_house_sales`: Verkäufe (Datum, Preis, `house_id`)
- `king_county_house_details`: Merkmale der Häuser (Fläche, Zimmer, Lage, Qualität, ...)

Die Zugangsdaten stehen in der Datei `.env` (Vorlage: `.env.example`) und liegen nicht im Repository.

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

## Verbindung zur Datenbank

In [2]:
load_dotenv(override=True)

url = URL.create(
    drivername="postgresql+psycopg2",
    host=os.getenv("HOST"),
    port=os.getenv("PORT"),
    database=os.getenv("DATABASE"),
    username=os.getenv("USER_DB"),
    password=os.getenv("PASSWORD"),
)

fehlend = [k for k in ["HOST", "PORT", "DATABASE", "USER_DB", "PASSWORD"] if not os.getenv(k)]
if fehlend:
    raise ValueError(f"In der .env fehlen: {fehlend}")

engine = create_engine(url)

## Tabellen verbinden und laden

Ein `FULL JOIN` über `house_id` und `id` verbindet Verkäufe und Hausmerkmale. Ein Haus, das mehrfach verkauft wurde, erscheint dabei einmal pro Verkauf.

In [3]:
query = """
    SELECT *
    FROM eda.king_county_house_sales s
    FULL JOIN eda.king_county_house_details d
        ON s.house_id = d.id;
"""

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print(df.shape)
df.head()

(21597, 23)


,date,price,house_id,id,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.0,7129300520,1,7129300520,3.0,1.00,1180.0,5650.0,1.0,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2014-12-09,538000.0,6414100192,2,6414100192,3.0,2.25,2570.0,7242.0,2.0,...,7,2170.0,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0
2,2015-02-25,180000.0,5631500400,3,5631500400,2.0,1.00,770.0,10000.0,1.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,2014-12-09,604000.0,2487200875,4,2487200875,4.0,3.00,1960.0,5000.0,1.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,2015-02-18,510000.0,1954400510,5,1954400510,3.0,2.00,1680.0,8080.0,1.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           21597 non-null  object 
 1   price          21597 non-null  float64
 2   house_id       21597 non-null  int64  
 3   id             21597 non-null  int64  
 4   id             21597 non-null  int64  
 5   bedrooms       21597 non-null  float64
 6   bathrooms      21597 non-null  float64
 7   sqft_living    21597 non-null  float64
 8   sqft_lot       21597 non-null  float64
 9   floors         21597 non-null  float64
 10  waterfront     19206 non-null  float64
 11  view           21534 non-null  float64
 12  condition      21597 non-null  int64  
 13  grade          21597 non-null  int64  
 14  sqft_above     21597 non-null  float64
 15  sqft_basement  21145 non-null  float64
 16  yr_built       21597 non-null  int64  
 17  yr_renovated   17749 non-null  float64
 18  zipcode        21

## Als CSV speichern

In [5]:
os.makedirs("data", exist_ok=True)
df.to_csv("data/eda.csv", index=False)
print("Gespeichert: data/eda.csv")

Gespeichert: data/eda.csv
